# Détection de fraudes par carte de crédit avec un auto-encodeur (PyTorch)

**Version 2 (corrigée).** Auto-encodeur dense entraîné **uniquement sur des transactions normales** :
une transaction frauduleuse, que le modèle n'a jamais apprise, est mal reconstruite, donc son
**erreur de reconstruction** est élevée. Un seuil sur cette erreur sépare les anomalies.

**Corrections par rapport à la version remise** (`project_ua1_remis.ipynb`) :

| Problème dans la version remise | Correction ici |
|---|---|
| Couche latente de 16 pour 7 variables (pas de goulot d'étranglement) | `d_latent` testé de 1 à 7, choix par la courbe de l'erreur de validation |
| Validation contenant des fraudes (early stopping et choix du modèle) | Validation **sur les normales seulement** ; les fraudes ne servent qu'à comparer/évaluer |
| Seul le batch size était testé | Recherche aléatoire : `lr` (échelle log), `weight_decay`, largeur |
| Comparaison classique/débruitant faussée (validation du débruitant bruitée) | Même validation propre pour les deux, choix sur la validation (jamais sur le test) |
| Seeds en commentaire, deux `random_state` différents | Une seule graine (`RANDOM_STATE = 42`) |
| Sauvegarde du `state_dict` seul | Sauvegarde du modèle + config + scaler + seuil, avec rechargement vérifié |
| Exemples d'anomalies : plus fortes erreurs, dont des fausses alertes | Vraies fraudes détectées, avec leurs valeurs et la variable responsable |

**Données :** `credit_card_data.csv` (1 000 000 de transactions, ~8,7 % de fraudes), non incluses dans le dépôt (voir le README).

## 1. Imports et configuration

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay,
                             precision_score, recall_score, f1_score)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Appareil utilisé : {device}")

# --- Configuration ---
DATA_PATH = "credit_card_data.csv"   # sur Colab : téléverser le fichier ou adapter le chemin
N_LIGNES = None                      # None = tout le jeu de données ; ex. 100_000 pour un test rapide
BATCH_SIZE = 512
EPOCHS_RECHERCHE = 15                # époques max pour les recherches (d_latent, hyperparamètres)
EPOCHS_FINAL = 40                    # époques max pour les modèles finaux
PATIENCE = 5                         # patience de l'early stopping
os.makedirs("checkpoints", exist_ok=True)

## 2. Chargement et exploration des données

In [ ]:
credit_card = pd.read_csv(DATA_PATH)
if N_LIGNES is not None:
    credit_card = credit_card.sample(N_LIGNES, random_state=RANDOM_STATE)

print("Dimensions :", credit_card.shape)
credit_card.head()

In [ ]:
print("Valeurs manquantes :", credit_card.isna().sum().sum())
print("Doublons :", credit_card.duplicated().sum())
print()
print(credit_card["fraud"].value_counts())
print(credit_card["fraud"].value_counts(normalize=True).round(4))

Les données sont **fortement déséquilibrées** (~8,7 % de fraudes). L'*accuracy* serait donc trompeuse :
un modèle qui répondrait toujours « normale » aurait environ 91 % d'accuracy sans détecter aucune fraude.
On utilise la **précision**, le **rappel** et le **F1-score**.

In [ ]:
continues = ["distance_from_home", "distance_from_last_transaction", "ratio_to_median_purchase_price"]
display(credit_card[continues].describe())

credit_card.hist(figsize=(16, 10))
plt.tight_layout()
plt.show()

Les trois variables continues ont des **échelles très différentes** et de **fortes valeurs extrêmes**
(asymétrie à droite : moyenne > médiane). On les standardise avant l'entraînement ; les quatre autres
variables sont binaires.

## 3. Prétraitement

- Séparation **70 % / 15 % / 15 %** stratifiée (`stratify=y`) : validation et test contiennent des fraudes dans la même proportion.
- **Entraînement et validation d'entraînement : transactions normales uniquement** (`fraud == 0`).
- `StandardScaler` ajusté **uniquement sur les normales d'entraînement**, puis appliqué tel quel au reste (pas de fuite d'information).
- La colonne `fraud` n'est **jamais** une entrée du modèle : elle sert à filtrer les normales et à évaluer.

In [ ]:
colonnes = [c for c in credit_card.columns if c != "fraud"]
X = credit_card[colonnes]
y = credit_card["fraud"].astype(int)

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_STATE)

print("Train :", X_train.shape, "| Validation :", X_val.shape, "| Test :", X_test.shape)
print("Fraudes dans le test :", int(y_test.sum()))

X_train_normal = X_train[y_train == 0]
X_val_normal = X_val[y_val == 0]
print("Normales pour l'entraînement :", X_train_normal.shape[0])
print("Normales de validation :", X_val_normal.shape[0])

In [ ]:
scaler = StandardScaler()
X_train_normal = scaler.fit_transform(X_train_normal)   # fit UNIQUEMENT sur les normales d'entraînement
X_val_normal = scaler.transform(X_val_normal)
X_val = scaler.transform(X_val)                         # validation complète (avec fraudes) : comparaison de modèles seulement
X_test = scaler.transform(X_test)

y_val_np = y_val.to_numpy()
y_test_np = y_test.to_numpy()

# float32 : format numérique standard de PyTorch
def vers_tenseur(a):
    return torch.tensor(a, dtype=torch.float32)

X_train_normal, X_val_normal, X_val, X_test = map(
    vers_tenseur, (X_train_normal, X_val_normal, X_val, X_test))

train_loader = DataLoader(TensorDataset(X_train_normal), batch_size=BATCH_SIZE, shuffle=True)
# Validation pour l'early stopping, le seuil et le choix des hyperparamètres : NORMALES SEULEMENT
val_normal_loader = DataLoader(TensorDataset(X_val_normal), batch_size=BATCH_SIZE, shuffle=False)
# Validation complète (avec fraudes) : uniquement pour comparer des types de modèles
val_complete_loader = DataLoader(TensorDataset(X_val), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(TensorDataset(X_test), batch_size=BATCH_SIZE, shuffle=False)

## 4. Auto-encodeur et fonctions d'entraînement

**Architecture** : entrée (7) → `largeur` → `largeur/2` → **code latent** → `largeur/2` → `largeur` → sortie (7).
- Couches **denses** avec **ReLU** (non-linéarité simple et standard) ; entonnoir pour forcer la compression.
- **Aucune activation en sortie** : les données standardisées peuvent être négatives ou dépasser 1, une sigmoïde ou une ReLU empêcherait de les reconstruire.
- Perte **MSE** : la cible est l'entrée elle-même (propre, même pour le débruitant).
- **Early stopping** : on garde les poids de la meilleure perte de validation (normales) et on les recharge à la fin.

In [ ]:
class AutoEncodeur(nn.Module):
    def __init__(self, inputs=7, d_latent=3, largeur=128):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(inputs, largeur), nn.ReLU(),
            nn.Linear(largeur, largeur // 2), nn.ReLU(),
            nn.Linear(largeur // 2, d_latent),
        )
        self.decoder = nn.Sequential(
            nn.Linear(d_latent, largeur // 2), nn.ReLU(),
            nn.Linear(largeur // 2, largeur), nn.ReLU(),
            nn.Linear(largeur, inputs),          # pas d'activation (données standardisées)
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))


def entrainer(model, optimizer, epochs, patience, nom_fichier,
              train_loader, val_loader, bruit=0.0):
    """Entraîne avec early stopping. bruit=0 : classique ; bruit>0 : débruitant.
    La validation est TOUJOURS calculée sur une entrée propre."""
    criterion = nn.MSELoss()
    model = model.to(device)
    train_losses, val_losses = [], []
    meilleure_val, compteur, meilleure_epoque = float("inf"), 0, 0

    for epoch in range(epochs):
        model.train()
        cumul = 0.0
        for (X_batch,) in train_loader:
            X_batch = X_batch.to(device)
            entree = X_batch + bruit * torch.randn_like(X_batch)   # bruit = 0 -> entrée inchangée
            optimizer.zero_grad()
            loss = criterion(model(entree), X_batch)               # cible : l'entrée propre
            loss.backward()
            optimizer.step()
            cumul += loss.item()
        train_losses.append(cumul / len(train_loader))

        model.eval()
        cumul = 0.0
        with torch.no_grad():
            for (X_batch,) in val_loader:
                X_batch = X_batch.to(device)
                cumul += criterion(model(X_batch), X_batch).item()
        val_losses.append(cumul / len(val_loader))

        if val_losses[-1] < meilleure_val:
            meilleure_val, compteur, meilleure_epoque = val_losses[-1], 0, epoch
            torch.save(model.state_dict(), nom_fichier)
        else:
            compteur += 1
            if compteur >= patience:
                break

    # Recharger les meilleurs poids (sinon model garde ceux de la dernière époque)
    model.load_state_dict(torch.load(nom_fichier, map_location=device))
    return train_losses, val_losses, meilleure_epoque


def calculer_erreurs(model, data_loader):
    """Erreur de reconstruction (MSE moyenne sur les 7 variables) de chaque transaction."""
    model.eval()
    erreurs = []
    with torch.no_grad():
        for (X_batch,) in data_loader:
            X_batch = X_batch.to(device)
            erreur = torch.mean((X_batch - model(X_batch)) ** 2, dim=1)
            erreurs.extend(erreur.cpu().numpy())
    return np.array(erreurs)

## 5. Choix de la taille de la couche latente

Une latente **plus grande que l'entrée** permet de recopier l'entrée sans rien apprendre. On entraîne un auto-encodeur
pour chaque `d_latent` de 1 à 7 et on trace l'erreur de reconstruction de validation (normales). On retient la **plus petite**
latente dont l'erreur reste à moins de 10 % du meilleur résultat (règle du « coude » ; le seuil de 10 % est un choix
pratique, on peut le modifier).

In [ ]:
dims = [1, 2, 3, 4, 5, 7]
erreurs_dim = {}

for d in dims:
    torch.manual_seed(RANDOM_STATE)            # même graine avant chaque modèle
    m = AutoEncodeur(7, d, 128)
    opt = torch.optim.Adam(m.parameters(), lr=0.001)
    _, val_l, _ = entrainer(m, opt, EPOCHS_RECHERCHE, PATIENCE,
                            f"checkpoints/d_latent_{d}.pt", train_loader, val_normal_loader)
    erreurs_dim[d] = min(val_l)
    print(f"d_latent = {d} : MSE de validation (normales) = {erreurs_dim[d]:.6f}")

plt.plot(list(erreurs_dim.keys()), list(erreurs_dim.values()), marker="o")
plt.xlabel("Taille de la couche latente")
plt.ylabel("MSE de validation (normales)")
plt.title("Erreur de reconstruction selon la taille du goulot")
plt.show()

meilleur = min(erreurs_dim.values())
D_LATENT = min(d for d, e in erreurs_dim.items() if e <= 1.10 * meilleur)
print("d_latent retenu :", D_LATENT)

## 6. Recherche aléatoire d'hyperparamètres

8 essais : `lr` et `weight_decay` tirés en **échelle logarithmique** (c'est le rapport qui compte, pas la différence), largeur
tirée dans {32, 64, 128}. Sélection sur la **MSE de validation des normales**, jamais sur le test.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
resultats = []

for i in range(8):
    lr = 10 ** rng.uniform(-4, -2)
    wd = 10 ** rng.uniform(-6, -3)
    largeur = int(rng.choice([32, 64, 128]))

    torch.manual_seed(RANDOM_STATE)
    m = AutoEncodeur(7, D_LATENT, largeur)
    opt = torch.optim.Adam(m.parameters(), lr=lr, weight_decay=wd)
    _, val_l, _ = entrainer(m, opt, EPOCHS_RECHERCHE, PATIENCE,
                            f"checkpoints/essai_{i}.pt", train_loader, val_normal_loader)
    resultats.append({"essai": i, "lr": lr, "weight_decay": wd,
                      "largeur": largeur, "val_mse": min(val_l)})
    print(f"Essai {i} : lr={lr:.5f} | wd={wd:.1e} | largeur={largeur} | MSE val={min(val_l):.6f}")

tableau = pd.DataFrame(resultats).sort_values("val_mse")
display(tableau)

meilleur_essai = tableau.iloc[0]
BEST_LR = float(meilleur_essai["lr"])
BEST_WD = float(meilleur_essai["weight_decay"])
BEST_LARGEUR = int(meilleur_essai["largeur"])
print("Hyperparamètres retenus :", BEST_LR, BEST_WD, BEST_LARGEUR)

## 7. Comparaison : auto-encodeur classique vs débruitant

Mêmes hyperparamètres, même graine, **même validation propre**. Le débruitant reçoit en entrée l'entrée + un bruit
gaussien (écart-type 0,1) et doit reconstruire l'entrée propre.

La comparaison se fait sur la **détection** (précision, rappel, F1 sur la **validation complète**, avec le seuil au 95e centile
des normales de validation), et non sur la MSE seule. Le **test n'est utilisé qu'une fois**, pour le modèle retenu.

In [ ]:
def seuil_95(model):
    return np.percentile(calculer_erreurs(model, val_normal_loader), 95)

def scores(y_true, pred):
    return (precision_score(y_true, pred), recall_score(y_true, pred), f1_score(y_true, pred))

modeles, lignes = {}, []
for nom, bruit in [("Classique", 0.0), ("Débruitant", 0.1)]:
    torch.manual_seed(RANDOM_STATE)
    m = AutoEncodeur(7, D_LATENT, BEST_LARGEUR)
    opt = torch.optim.Adam(m.parameters(), lr=BEST_LR, weight_decay=BEST_WD)
    tl, vl, ep = entrainer(m, opt, EPOCHS_FINAL, PATIENCE,
                           f"checkpoints/{nom}.pt", train_loader, val_normal_loader, bruit=bruit)
    s = seuil_95(m)
    pred_val = (calculer_erreurs(m, val_complete_loader) > s).astype(int)
    p, r, f = scores(y_val_np, pred_val)
    modeles[nom] = {"model": m, "seuil": s, "f1_val": f, "train": tl, "val": vl}
    lignes.append({"modèle": nom, "meilleure époque": ep + 1, "MSE val (normales, propre)": min(vl),
                   "seuil": s, "précision (val)": p, "rappel (val)": r, "F1 (val)": f})

comparaison = pd.DataFrame(lignes)
display(comparaison)

nom_final = max(modeles, key=lambda n: modeles[n]["f1_val"])
modele_final = modeles[nom_final]["model"]
seuil = modeles[nom_final]["seuil"]
print("Modèle retenu (meilleur F1 de validation) :", nom_final)

plt.figure(figsize=(8, 4))
for nom in modeles:
    plt.plot(modeles[nom]["val"], label=f"{nom} (validation, entrée propre)")
plt.xlabel("Époque"); plt.ylabel("MSE de validation"); plt.legend()
plt.title("Courbes de validation des deux auto-encodeurs")
plt.show()

## 8. Détection d'anomalies et évaluation finale (jeu de test)

**Seuil** : 95e centile des erreurs de reconstruction des **transactions normales de validation**, sans regarder les fraudes.
Par construction, environ 5 % des normales seront signalées à tort (fausses alertes). Monter le seuil réduit les fausses
alertes (meilleure précision) mais fait manquer plus de fraudes (rappel plus bas) ; le baisser fait l'inverse.

In [ ]:
erreurs_test = calculer_erreurs(modele_final, test_loader)
predictions = (erreurs_test > seuil).astype(int)    # 1 = anomalie, 0 = normale

p, r, f = scores(y_test_np, predictions)
print(f"Seuil de détection : {seuil:.6f}")
print(f"Précision : {p:.3f} | Rappel : {r:.3f} | F1 : {f:.3f}")

cm = confusion_matrix(y_test_np, predictions)
print(cm)
ConfusionMatrixDisplay(cm, display_labels=["Normale", "Fraude"]).plot()
plt.title(f"Matrice de confusion - {nom_final}")
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(erreurs_test[y_test_np == 0], bins=100, alpha=0.7, label="Transactions normales")
plt.hist(erreurs_test[y_test_np == 1], bins=100, alpha=0.7, label="Fraudes")
plt.axvline(seuil, linestyle="--", color="black", label=f"Seuil = {seuil:.6f}")
plt.xscale("log")
plt.xlabel("Erreur de reconstruction (échelle log)")
plt.ylabel("Nombre de transactions")
plt.title("Distribution des erreurs de reconstruction (test)")
plt.legend()
plt.show()

### Exemples concrets de fraudes correctement détectées

On prend les **vraies fraudes** (`fraud = 1`) dont l'erreur dépasse le seuil, les plus fortes d'abord. Pour chacune on
affiche les valeurs d'origine et la **variable qui contribue le plus** à l'erreur (interprétation simple : quelle variable
le modèle n'arrive pas à reconstruire).

In [ ]:
X_test_np = X_test.numpy()
modele_final.eval()
with torch.no_grad():
    reconstruction = modele_final(X_test.to(device)).cpu().numpy()
erreur_par_variable = (X_test_np - reconstruction) ** 2

vrais_positifs = np.where((predictions == 1) & (y_test_np == 1))[0]
top = vrais_positifs[np.argsort(erreurs_test[vrais_positifs])[-5:][::-1]]

exemples = pd.DataFrame(scaler.inverse_transform(X_test_np[top]), columns=colonnes)
exemples["erreur"] = erreurs_test[top]
exemples["variable_principale"] = [colonnes[j] for j in erreur_par_variable[top].argmax(axis=1)]

print(f"Seuil = {seuil:.6f}  |  vraies fraudes détectées : {len(vrais_positifs)}")
display(exemples.round(3))
print("Valeurs moyennes des transactions normales (référence) :")
display(pd.Series(scaler.mean_, index=colonnes).round(3))

## 9. Sauvegarde du modèle final et rechargement

Le fichier `.pt` contient les poids **et** ce qu'il faut pour réutiliser le modèle : architecture, seuil, moyenne et
écart-type du scaler. On vérifie ensuite que le modèle rechargé donne les mêmes erreurs.

In [ ]:
checkpoint = {
    "state_dict": modele_final.state_dict(),
    "type": nom_final,
    "d_latent": D_LATENT,
    "largeur": BEST_LARGEUR,
    "seuil": float(seuil),
    "colonnes": colonnes,
    "moyenne": scaler.mean_.tolist(),
    "ecart_type": scaler.scale_.tolist(),
}
torch.save(checkpoint, "autoencodeur_final.pt")
print("Modèle sauvegardé : autoencodeur_final.pt")

ckpt = torch.load("autoencodeur_final.pt", map_location=device, weights_only=True)
modele_recharge = AutoEncodeur(7, ckpt["d_latent"], ckpt["largeur"]).to(device)
modele_recharge.load_state_dict(ckpt["state_dict"])
modele_recharge.eval()

erreurs_recharge = calculer_erreurs(modele_recharge, test_loader)
print("Mêmes erreurs après rechargement :", np.allclose(erreurs_test, erreurs_recharge, atol=1e-6))

In [ ]:
def detecter(df_transactions, chemin="autoencodeur_final.pt"):
    """Réutilise le modèle sauvegardé sur de NOUVELLES transactions (DataFrame brut, mêmes 7 colonnes).
    Retourne l'erreur de reconstruction et 1 (anomalie) / 0 (normale) pour chaque ligne."""
    ck = torch.load(chemin, map_location=device, weights_only=True)
    m = AutoEncodeur(7, ck["d_latent"], ck["largeur"]).to(device)
    m.load_state_dict(ck["state_dict"])
    m.eval()

    X_brut = df_transactions[ck["colonnes"]].to_numpy(dtype=np.float64)
    X_std = (X_brut - np.array(ck["moyenne"])) / np.array(ck["ecart_type"])
    X_t = torch.tensor(X_std, dtype=torch.float32).to(device)
    with torch.no_grad():
        err = torch.mean((X_t - m(X_t)) ** 2, dim=1).cpu().numpy()
    return err, (err > ck["seuil"]).astype(int)

# Exemple : 5 transactions du jeu de test (valeurs brutes)
exemple_brut = pd.DataFrame(scaler.inverse_transform(X_test_np[:5]), columns=colonnes)
err, flag = detecter(exemple_brut)
print(err, flag)

## Conclusion

L'auto-encodeur est entraîné uniquement sur des transactions normales ; l'erreur de reconstruction identifie les
transactions atypiques, avec un seuil au 95e centile des erreurs des normales de validation. Le modèle final est sauvegardé
avec sa configuration, son scaler et son seuil, puis rechargé et vérifié.

**Ce qui fonctionne** : les fraudes qui s'écartent nettement du comportement normal (valeurs extrêmes) sont mal reconstruites
et dépassent le seuil.

**Limites** : une erreur élevée signifie « atypique », pas toujours « fraude » (fausses alertes) ; les fraudes qui ressemblent
à des transactions normales sont reconstruites correctement et manquées ; le seuil au 95e centile impose environ 5 % de fausses
alertes sans être optimisé selon le coût d'une fraude manquée ; la MSE est sensible aux valeurs extrêmes des variables continues.

**Pistes d'amélioration** : transformation `log1p` des variables asymétriques avant standardisation, auto-encodeur épars,
optimisation bayésienne (Optuna) des hyperparamètres, seuil choisi selon le compromis précision/rappel voulu
(courbe précision-rappel), régularisation supplémentaire.